# jpoke.testing のヘルパーを使い、これまでのサンプルで繰り返し書いてきた
Battle(...).start() + Player.add_pokemon() の定型処理を1呼び出しにまとめる方法を示す

jpoke.testing の位置づけ（内部テスト専用ヘルパーからの昇格）・提供するヘルパー一覧は
docs/api/README.md の テストユーティリティ（jpoke.testing）節を参照。
「状態を変えて比較する」シナリオ検証コードが短く書けるようになる。

Google Colabで開いた場合は、まず次のセルで `jpoke` をインストールする
（ローカルで `pip install jpoke` 済みなら不要）。

In [ ]:
%pip install -q jpoke

In [ ]:
# TODO: testing helper は公開しない。このexampleも削除
from __future__ import annotations

from jpoke import Pokemon
from jpoke.testing import apply_ailment, calc_lethal, run_move, start_battle

In [ ]:
move_name = "ドラゴンテール"

# start_battle()の挙動（accuracy等のキーワード引数含む）は
# docs/api/README.md の テストユーティリティ節を参照
battle = start_battle(
    team0=[Pokemon("ガブリアス", move_names=[move_name])],
    team1=[Pokemon("カイリュー", item_name="オボンのみ")],
    accuracy=100,
)
# start_battle() は Player インスタンスを返さないため、battle.players から取得する
# （詳細は docs/api/README.md の Battle「対戦進行系」節の players 属性を参照）
player0, player1 = battle.players
defender = battle.get_active(player1)

plain_final = calc_lethal(battle, player_idx=0, moves=move_name, max_attack=2)[-1]

In [ ]:
apply_ailment(battle, player_idx=1, ailment_name="どく")
print(f"防御側の状態異常: {defender.status}")

poisoned_final = calc_lethal(battle, player_idx=0, moves=move_name, max_attack=2)[-1]
print(
    f"{move_name}を{plain_final.n_attack}発当てた時点の致死率: "
    f"どく無し {plain_final.lethal_probability:.2%} / "
    f"どくあり {poisoned_final.lethal_probability:.2%}"
)

In [ ]:
# run_move()の挙動は docs/api/README.md の テストユーティリティ節を参照
print("-" * 50)
hp_before = defender.hp
run_move(battle, player_idx=0, move_idx=0)
print(f"{move_name}を実際に撃ち込んだ後、{defender.name}のHP: {hp_before} → {defender.hp}")

試してみよう: apply_ailment() の ailment_name を別の状態異常に変えたり、
start_battle() の weather / terrain / side0 / side1 引数で場の状態を
追加してから calc_lethal() を呼ぶと、致死率がどう変わるか比較できる